In [64]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from mlxtend.feature_selection import SequentialFeatureSelector 
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from sklearn.pipeline import Pipeline, make_pipeline

import warnings
warnings.filterwarnings('ignore')

**The data provided are responses for a survey conducted in 2015 on a sample of 70 thousand people to assess the factors leading to getting diagnosed with diabetes.**

In [3]:
df = pd.read_csv("diabetes.csv")
df.head()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,3.0,5.0,30.0,0.0,1.0,4.0,6.0,8.0
1,0.0,1.0,1.0,1.0,26.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,12.0,6.0,8.0
2,0.0,0.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,10.0,0.0,1.0,13.0,6.0,8.0
3,0.0,1.0,1.0,1.0,28.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,0.0,3.0,0.0,1.0,11.0,6.0,8.0
4,0.0,0.0,0.0,1.0,29.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,8.0,5.0,8.0



### Exploratory Data Analysis ###

In [4]:
df.shape

(70692, 22)

In [5]:
# taking a random sample of 12000 record to make the process easier 
df = df.sample(12000, random_state=49)

In [6]:
# checking for null values
df.isna().sum()

Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

In [7]:
# exploring columns
df.columns

Index(['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income'],
      dtype='object')

In [8]:
# exploring the unique values of each column
for col in df.columns:
    print(f'{col}:', df[col].unique())

Diabetes_binary: [0. 1.]
HighBP: [0. 1.]
HighChol: [0. 1.]
CholCheck: [1. 0.]
BMI: [24. 28. 22. 42. 31. 49. 26. 33. 34. 30. 32. 25. 21. 29. 19. 39. 43. 37.
 47. 23. 36. 35. 41. 27. 40. 55. 20. 38. 44. 48. 45. 18. 46. 50. 87. 51.
 67. 15. 58. 73. 53. 54. 63. 75. 17. 61. 52. 16. 59. 77. 79. 57. 13. 82.
 72. 66. 60. 92. 84. 65. 64. 62. 56. 89. 81. 14. 71.]
Smoker: [0. 1.]
Stroke: [0. 1.]
HeartDiseaseorAttack: [0. 1.]
PhysActivity: [1. 0.]
Fruits: [1. 0.]
Veggies: [1. 0.]
HvyAlcoholConsump: [0. 1.]
AnyHealthcare: [1. 0.]
NoDocbcCost: [1. 0.]
GenHlth: [2. 4. 5. 3. 1.]
MentHlth: [ 0. 30.  1.  8.  5. 10. 15. 20.  2. 25. 12.  4. 29.  3.  7.  6. 14. 21.
 18. 23. 13. 17. 26. 19. 28. 27.  9. 16. 11. 22. 24.]
PhysHlth: [ 2. 10. 30. 14.  0.  3.  1.  5. 15.  4.  8. 20.  7. 18. 29.  6.  9. 28.
 25. 12. 16. 27. 21. 13. 17. 24. 11. 22. 23. 26. 19.]
DiffWalk: [0. 1.]
Sex: [0. 1.]
Age: [ 6. 10. 13.  8.  9. 12. 11.  7.  4.  1.  3.  5.  2.]
Education: [6. 4. 3. 5. 2. 1.]
Income: [5. 1. 4. 7. 6. 8. 3. 2.]


In [9]:
# exploring the distribution of BMI
fig = px.histogram(df, x='BMI', title= 'BMI Distribution')
fig.update_layout(template='plotly_white', yaxis_title=None)
fig.show()

In [10]:
fig = px.box(df, x='BMI', title= 'BMI Distribution')
fig.update_layout(template='plotly_white', yaxis_title=None)
fig.show()

In [11]:
# visualizing the respondents answer to the question "In the previous month, how many days have you felt that your mental health was bad"
fig = px.box(df, x='MentHlth', title= 'How many days in the previous month your mental health was bad', labels={'MentHlth': 'Mental Health'})
fig.update_layout(template='plotly_white', yaxis_title=None)
fig.show()

In [12]:
# visualizing the respondents answer to the question "In the previous month, how many days have you felt that your physical health was bad"

fig = px.box(df, x='PhysHlth', title= 'How many days in the previous month your physical health was bad', labels={'PhysHlth': 'Physical Health'})
fig.update_layout(template='plotly_white', yaxis_title=None)
fig.show()

In [13]:
# create a copy of the data to categorize the numerical columns to visualize them better
df_c = df.copy()

# map categories for each numerical value
df_c['Sex'] = df_c['Sex'].map({0: 'Female', 1: 'Male'})
bin_cols = ['Diabetes_binary', 'HighBP', 'HighChol', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity',
       'HvyAlcoholConsump']

# mapping binary columns
for col in bin_cols:
    df_c[col] = df_c[col].map({0: 'No', 1:'Yes'})
# concatenating the column 'Sex' to the resto of the dataframe
bin_cols = bin_cols + ['Sex']

In [14]:
# customize titles for each binary column
titles = ['Diabetes', 'High blood pressure', 'High cholestrol', 'Smoker', 'Stroke', 'Heart Disease/Attack', 'Physical Activity',
          'Alcholicism', 'Sex']

# create a 3x3 subplot to visualize the binary columns using pie charts
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=titles,
    specs=[[{'type': 'domain'}]*3]*3  # each cell holds a pie chart
    )

# loop through each column to add it to the subplot with a customized title and percentage of each value
for i, (col, title) in enumerate(zip(bin_cols, titles)):
    counts = df_c[col].value_counts().reset_index()
    counts.columns = [col, 'Count']

    row = i // 3 + 1
    col_pos = i % 3 + 1

    # add the figure to the subplot
    fig.add_trace(
        go.Pie(labels=counts[col], values=counts['Count'], textinfo='percent+label'),
        row=row, col=col_pos
    )
# customize the graph
fig.update_layout(
    height=1100,
    title_text='Distribution of Binary Columns',
    template='plotly_white',
    showlegend=False
)

fig.show()


In [15]:
# create a list of factors to get a better look at their impact on the diabetes diagnosis
factors = ['HighBP', 'HighChol', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'HvyAlcoholConsump']

# create a list of titles for the factors 
fac_titles = ['High blood pressure', 'High cholestrol', 'Smoker', 'Stroke', 'Heart Disease/Attack', 'Physical Activity', 'Alcholic']

# loop through each factor and visualize it with a stacked bar chart to show the percentage of each category
for col, title in zip(bin_cols, fac_titles):
    fig = px.histogram(df_c, x=col, color='Diabetes_binary', histnorm='percent',
     labels={f'{col}': f'{title}', 'Diabetes_binary': 'Diabetic'}, title=title)
    fig.update_layout(template='plotly_white', yaxis_title=None, xaxis_title=title)
    fig.show()

In [16]:
# mapping categories into general health column
df_c['GenHlth'] = df_c['GenHlth'].map({1: 'Excellent', 2: 'Very Good', 3: 'Good', 4: 'Fair', 5: 'Poor'})

# declaring the order of the general health
health_order = ['Excellent', 'Very Good', 'Good', 'Fair', 'Poor']
df_c['GenHlth'] = pd.Categorical(df_c['GenHlth'], categories=health_order, ordered=True)
counts = df_c['GenHlth'].value_counts().sort_index().reset_index()
counts.columns = ['GenHlth', 'counts']

# visualize the answer via horizontal bar chart
fig = px.bar(counts, x='counts', y='GenHlth', orientation='h', color='GenHlth', labels={'GenHlth': 'General Health'})
fig.update_layout(template='plotly_white', showlegend=False, xaxis_title=None)
fig.show()

In [17]:
# mapping age bins into the age column and sorting them ascendingly 
df_c['Age'] = df_c['Age'].map({1: '18-24', 2: '25-29', 3: '30-34', 4: '35-39', 5: '40-44', 6: '45-49', 7: '50-54',8: '55-59', 9: '60-64', 10: '65-69', 11: '70-74', 12: '75-79', 13: '80+'})
order = ['18-24','25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80+']

df_c['Age'] = pd.Categorical(df_c['Age'], categories=order, ordered=True)
counts = df_c['Age'].value_counts().sort_index().reset_index()
counts.columns = ['Age', 'count']

# create a bar chart for age bins
fig = px.bar(counts, x='Age', y='count', title='Age Distribution')
fig.update_layout(template='plotly_white', showlegend=False, yaxis_title=None)
fig.show()

In [18]:
# mapping education levels to the numerical column
edu = {1: 'Never Attended', 2: 'Elementary', 3: 'High School dropout', 4: 'High School graduate', 5: 'College dropout', 6: 'College graduate'}
order = ['Never Attended', 'Elementary', 'High School dropout', 'High School graduate', 'College dropout', 'College graduate']
df_c['Education'] = df_c['Education'].map(edu)

df_c['Education'] = pd.Categorical(df_c['Education'], categories=order, ordered=True)
counts = df_c['Education'].value_counts().sort_index().reset_index()
counts.columns = ['Education', 'Count']

# create a bar chart 
fig = px.bar(counts, x='Education', y='Count', color='Education')
fig.update_layout(template='plotly_white', showlegend=False, yaxis_title=None)
fig.show()

### Dimensionality Reduction ###

Preprocessing

In [19]:
# create a correlation matrix to possibily eliminate features that are highly correlated 
corr_matrix = df.corr()

fig = go.Figure(data=go.Heatmap(z=corr_matrix.values, x=corr_matrix.columns, y=corr_matrix.columns,
    colorscale='RdBu', zmin=-1, zmax=1, colorbar=dict(title="Correlation")))

fig.update_layout(title='Correlation Heatmap', xaxis_nticks=36, width=900, height=900)
fig.show()

In [81]:
# split the data into X: independent variables and y: target/dependent variable
X = df.drop('Diabetes_binary', axis=1)
y = df['Diabetes_binary']

In [82]:
# split the data into taining and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=60)

**Forward Feature Selection**

In [ ]:
# Perform forward feature selection to choose the best 6 features according to the accuracy of the SVM model
model = SVC()
forward_feature_selection = SequentialFeatureSelector(model,
k_features=6,
forward=True,
floating=False,
verbose=2,
scoring='accuracy',
cv=5).fit(X_train, y_train)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    5.6s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  21 out of  21 | elapsed:  2.5min finished

[2025-05-02 09:17:11] Features: 1/6 -- score: 0.6892857142857143[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    5.5s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  20 out of  20 | elapsed:  2.0min finished

[2025-05-02 09:19:11] Features: 2/6 -- score: 0.7117857142857142[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    5.3s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  19 out of  19 | elapsed:  1.8min finished

[2025-05-02 09:20:59] Features: 3/6 -- score: 0.7285714285714285[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 

In [ ]:
# print the list of 6 features
forward_feature_selection.k_feature_names_

('HighBP', 'HighChol', 'BMI', 'GenHlth', 'Age', 'Income')

In [ ]:
# print the model accuracy trained only on 6 features
forward_feature_selection.k_score_

0.7476190476190476

**Backward Feature Elimination**

In [70]:
backward_feature_selection = SequentialFeatureSelector(model,
k_features=10,
forward=False,
floating=False,
verbose=2,
scoring='accuracy',
cv=5).fit(X_train, y_train)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    6.3s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  21 out of  21 | elapsed:  3.2min finished

[2025-05-02 14:04:44] Features: 20/10 -- score: 0.7489285714285715[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:   13.3s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  20 out of  20 | elapsed:  3.5min finished

[2025-05-02 14:08:14] Features: 19/10 -- score: 0.7495238095238095[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:   11.7s remaining:    0.0s
[Parallel(n_jobs=1)]: Done  19 out of  19 | elapsed:  3.5min finished

[2025-05-02 14:11:44] Features: 18/10 -- score: 0.7494047619047619[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Don

In [73]:
metrics = pd.DataFrame.from_dict(backward_feature_selection.get_metric_dict()).T
metrics

,feature_idx,cv_scores,avg_score,feature_names,ci_bound,std_dev,std_err
21,"(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0.7380952380952381, 0.7583333333333333, 0.753...",0.745833,"(HighBP, HighChol, CholCheck, BMI, Smoker, Str...",0.01404,0.010924,0.005462
20,"(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0.7404761904761905, 0.7583333333333333, 0.756...",0.748929,"(HighBP, HighChol, CholCheck, BMI, Smoker, Str...",0.014422,0.011221,0.00561
19,"(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0.7410714285714286, 0.7559523809523809, 0.757...",0.749524,"(HighBP, HighChol, CholCheck, BMI, Smoker, Str...",0.013005,0.010118,0.005059
18,"(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[0.7422619047619048, 0.7571428571428571, 0.757...",0.749405,"(HighBP, HighChol, CholCheck, BMI, Smoker, Str...",0.011643,0.009059,0.004529
17,"(0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 17...","[0.7440476190476191, 0.7595238095238095, 0.756...",0.750357,"(HighBP, HighChol, CholCheck, BMI, Smoker, Str...",0.012313,0.00958,0.00479
16,"(0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 12, 13, 17, 1...","[0.743452380952381, 0.7583333333333333, 0.7583...",0.750714,"(HighBP, HighChol, CholCheck, BMI, Smoker, Str...",0.0121,0.009415,0.004707
15,"(0, 1, 2, 3, 4, 5, 7, 8, 10, 11, 13, 17, 18, 1...","[0.743452380952381, 0.7583333333333333, 0.7577...",0.750476,"(HighBP, HighChol, CholCheck, BMI, Smoker, Str...",0.011866,0.009232,0.004616
14,"(0, 1, 2, 3, 4, 5, 8, 10, 11, 13, 17, 18, 19, 20)","[0.7440476190476191, 0.7583333333333333, 0.758...",0.750476,"(HighBP, HighChol, CholCheck, BMI, Smoker, Str...",0.012378,0.00963,0.004815
13,"(0, 1, 2, 3, 4, 5, 10, 11, 13, 17, 18, 19, 20)","[0.743452380952381, 0.7583333333333333, 0.7577...",0.750238,"(HighBP, HighChol, CholCheck, BMI, Smoker, Str...",0.012361,0.009617,0.004809
12,"(0, 1, 2, 3, 4, 10, 11, 13, 17, 18, 19, 20)","[0.7446428571428572, 0.756547619047619, 0.7577...",0.750476,"(HighBP, HighChol, CholCheck, BMI, Smoker, Hvy...",0.010671,0.008303,0.004151


In [75]:
metrics.iloc[5, 3]

('HighBP',
 'HighChol',
 'CholCheck',
 'BMI',
 'Smoker',
 'Stroke',
 'PhysActivity',
 'Fruits',
 'HvyAlcoholConsump',
 'AnyHealthcare',
 'NoDocbcCost',
 'GenHlth',
 'Sex',
 'Age',
 'Education',
 'Income')

In [83]:
X_train = X_train[['HighBP', 'HighChol',
 'CholCheck', 'BMI',
 'Smoker','Stroke','PhysActivity',
 'Fruits','HvyAlcoholConsump',
 'AnyHealthcare','NoDocbcCost', 'GenHlth', 'Sex',
 'Age','Education', 'Income']]

X_test = X_test[['HighBP', 'HighChol',
 'CholCheck', 'BMI',
 'Smoker','Stroke','PhysActivity',
 'Fruits','HvyAlcoholConsump',
 'AnyHealthcare','NoDocbcCost', 'GenHlth', 'Sex',
 'Age','Education', 'Income']]

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

In [84]:
pca = PCA(n_components=0.95)
X_train = pca.fit_transform(X_train)
X_test = pca.fit_transform(X_test)

In [85]:
k_fold = KFold(n_splits=5)
scores = cross_val_score(model, X_train, y_train, cv = k_fold)
print("Cross Validation Scores: ", scores)
print("Average CV Score: ", scores.mean())

Cross Validation Scores:  [0.70416667 0.72142857 0.71845238 0.69285714 0.72678571]
Average CV Score:  0.7127380952380953


In [80]:
param_grid = [
    {'kernel': ['linear'], 'C': [0.1, 1, 10]},
    {'kernel': ['rbf'], 'C': [0.1, 1, 10], 'gamma': [0.1, 1, 'scale']},
    {'kernel': ['poly'], 'C': [0.1, 1, 10], 'degree': [2, 3]}
]

grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

grid_search.best_score_

0.7509523809523809

In [60]:
X_train = X_train[['HighBP', 'HighChol', 'BMI', 'GenHlth', 'Age', 'Income']]
X_train.head()

,HighBP,HighChol,BMI,GenHlth,Age,Income
34806,1.0,0.0,25.0,2.0,11.0,7.0
50262,1.0,1.0,45.0,3.0,7.0,8.0
9244,0.0,1.0,31.0,3.0,8.0,6.0
50638,1.0,1.0,25.0,2.0,6.0,8.0
43994,1.0,1.0,40.0,2.0,11.0,6.0


In [61]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

In [62]:
pca = PCA(n_components=0.95)
X_train = pca.fit_transform(X_train)
X_test = pca.fit_transform(X_test)

In [65]:
k_fold = KFold(n_splits=5)
scores = cross_val_score(model, X_train, y_train, cv = k_fold)
print("Cross Validation Scores: ", scores)
print("Average CV Score: ", scores.mean())

Cross Validation Scores:  [0.7452381  0.75178571 0.75357143 0.73988095 0.75119048]
Average CV Score:  0.7483333333333333


In [66]:
param_grid = [
    {'kernel': ['linear'], 'C': [0.1, 1, 10]},
    {'kernel': ['rbf'], 'C': [0.1, 1, 10], 'gamma': [0.1, 1, 'scale']},
    {'kernel': ['poly'], 'C': [0.1, 1, 10], 'degree': [2, 3]}
]

grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

grid_search.best_score_

0.7479761904761906